# பாடம் 04 - கருவி பயன்பாட்டு வடிவமைப்பு முறை

இந்த பாடத்தில் நீங்கள் மைக்ரோசாஃப்ட் அஞ்சன் ஃப்ரேம்வொர்க்கை (Python) பயன்படுத்தி AI முகவர்களுக்கான **கருவி பயன்பாடு** வடிவமைப்பு முறையை கற்றுக் கொள்வீர்கள். நாம் கையாளும் விஷயங்கள்:

- `@tool` அலங்கரிப்புடன் மற்றும் typed பரிமாற்றங்களுடன் செயல்பாட்டு கருவிகளை நிர்ணயித்தல்
- மாதிரி ஒவ்வொரு கருவி என்ன செய்கிறது என்பதை அறிய கருவி ஸ்கீமைகளை வழங்குதல்
- `approval_mode` மூலம் கருவி செயல்பாட்டை கட்டுப்படுத்தல்
- Pydantic மாதிரிகள் மற்றும் `response_format` மூலம் **கட்டமைக்கப்பட்ட வெளியீடு** வழங்கல்

காட்சியளிப்பு ஒரு **பயண முன்பதிவு முகவர்** ஆகும், இது இலக்குகளை தேடக்கூடியது, கிடைக்கும் இருப்பதை சரிபார்க்கக்கூடியது மற்றும் விமான தகவலைப் பெறக்கூடியது.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool அலங்காரி மூலம் கருவிகளைக் குறித்தல்

`@tool` அலங்காரி ஒரு சாதாரண Python செயல்பாட்டைப் பொறியாக்கி, ஒரு முகவர்கள் அழைக்கக்கூடிய கருவியாக மாற்றுகிறது.
முக்கிய அம்சங்கள்:

- **docstring** என்பது மாதிரிக்கு காட்சி அளிக்கப்படும் கருவி விளக்கமாகும்.
- **Type annotations** (விளக்கங்களுடன் `Annotated` உட்பட) கருவி வடிவமைப்பை வரையறுக்கிறது.
- `approval_mode` ஒரு அழைப்பை செயல்படுத்துவதற்கு முன்னர் பயனர் ஒப்புதல் தரவேண்டுமா என்பதை கட்டுப்படுத்துகிறது.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## பல கருவிகளுடன் ஒரு முகவரியை உருவாக்குதல்

மாடல் பயனரின் கேள்விக்கு பதிலளிக்க தேவையான எந்த கருவியையும் அழைக்க முடியும் என கிளையண்டுக்கு மூன்று கருவிகளையும் அனைத்து கொண்டு செல்லவும்.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## கருவிகள் உடன் அமைக்கப்பட்ட வெளியீடு

`response_format` ஐ Pydantic மாதிரியாக அமைப்பதினொடு, முகவர் சுதந்திரமாக உள்ளடக்கத்தைக் காட்டுவதற்கு பதிலாக நன்கு-வகைப்படுத்தப்பட்ட JSON பொருத்தத்தை திருப்ப வலியுறுத்தப்படுகிறது. இது கீழ்நிலை குறியீடு முடிவுகளை நிரல்பூர்வமாக பயன்படுத்த வேண்டிய போது பயனுள்ளதாக இருக்கும்.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## கருவி ஒப்புதல் எடுத்துக்காட்டு

`@tool` இல் உள்ள `approval_mode` நிலைமுறை கருவி அழைப்புகள் இயங்குவதற்கு முன் மனித ஒப்புதல் தேவைப்படுமா என்பதைக் கட்டுப்படுத்துகிறது:

| நிலைமுறை | நடத்தை |
|---|---|
| `"never_require"` | கருவி தானாக இயங்கும் — பயனர் உறுதிப்படுத்தல் தேவையில்லை. |
| `"always_require"` | ஒவ்வொரு அழைப்பும் இயங்குவதற்கு முன்னரும் பயனர் ஒப்புதல் பெறவேண்டும். |

புற விளைவுகள் உள்ள கருவிகளுக்கு (எ.கா., விமானத்தை முன்பதிவு செய்தல், கிரெடிட் கார்டை வசூலித்தல்) `"always_require"` ஐ பயன்படுத்தவும், இதனால் மனிதர் தொடர்ச்சியாக செயலின் உள்ளிருப்பில் இருக்கிறார்.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டது:

1. **கருவிகளை வரையறுக்க** `@tool` அபர்ணுனையை பயன்படுத்தி, வகைப்படுத்தப்பட்ட அளவுருக்களும் கருவி ஸ்கீமாகும் ஆவணப்பதிவுகளும் சேர்த்து.
2. **பல கருவிகளை ஒன்றிணைக்க** இயந்திரம் அவற்றை முறையே அழைத்து சிக்கலான கேள்விகளுக்கு பதில் அளிக்க.
3. **ஒழுங்குபடுத்தப்பட்ட வெளியீட்டை வழங்க** `response_format` ஆக Pydantic மாதிரியை அனுப்பி.
4. **கருவி ஒப்புதலை கட்டுப்படுத்த** `approval_mode` மூலம் நுண்ணறிவு செயல்பாடுகளுக்கு மனிதர் ஈடுபடச் செய்வது.

இந்த மாதிரிகள் நம்பகமான, தயாரிப்பு-தயார் செய்யப்பட்ட, வெளிப்புற அமைப்புகளுடன் பாதுகாப்பாக தொடர்பு கொள்ளக்கூடிய இயந்திரங்களை உருவாக்க அடிப்படையாகும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**உறுதிப்பத்திரம்**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியமாக்க முயற்சித்தாலும், இயந்திர மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனிக்கவும். உரையாடல் ஆவணத்தின் தாய்மொழியின்மேல் உள்ள கிடைப்பது அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவலுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படும் தவறான புரிதல்கள் அல்லது வழக்கறிஞர்கள் உடைய பொறுப்பேற்புகள் நாங்கள் ஏற்றுக்கொள்ளமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
